In [ ]:
%pip install --target ./pydeps memory_profiler

In [ ]:
import memory_profiler
print(memory_profiler.__version__)

In [ ]:
# ==================
# Modern ISD for low-weight codeword search: BJMM (Becker-Joux-May-Meurer,
# EUROCRYPT 2012), depth-2 "representation technique" variant.
# With eps = 0 it degenerates to MMT (May-Meurer-Thomae, ASIACRYPT 2011).
#

#
# 
#   Stern: u = u1 + u2, u1 supported on the left half of rows, u2 on the
#          right half, each of weight p/2  ->  a unique decomposition.
#   BJMM : u = u1 + u2 with u1, u2 supported on ALL k rows, each of weight
#          p1 = p/2 + eps  (the 2*eps common positions cancel).  Now u has
#          MANY decompositions ("representations"):
#              R = binom(p, p/2) * binom(k - p, eps).
#          We may therefore impose ~log2(R) extra linear constraints on
#          u1 (u1A = r on a sub-window B2 of size l2, r random) and still
#          expect one representation to survive; this shrinks the lists.
#          Each u1 is itself built Stern-style (two disjoint halves of rows,
#          collision on B2).  This is the depth-2 search tree of BJMM/MMT.
#
#   Rows are handled as Python ints (bit masks) for speed.

import os, time, traceback, multiprocessing
from sage.all import GF, PolynomialRing, matrix, vector

F2 = GF(2)
R = PolynomialRing(F2, 'x')
x = R.gen()

def vec(v):
    return vector(F2, v)

def hamming_weight_vec(v):
    return sum(int(b) for b in v)

def list_to_poly(L):
    return sum((c % 2) * x**i for i, c in enumerate(L))

def read_P_file(p_path):
    """Read instances (polynomials P as coefficient lists, low -> high) from file."""
    P_list = []
    with open(p_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            s = line.strip()
            if not s:
                continue
            if not (s.startswith("[") and s.endswith("]")):
                raise ValueError(f"Line {line_no}: invalid format")
            inner = s[1:-1].strip()
            P_list.append([] if inner == "" else [int(t.strip()) for t in inner.split(",")])
    return P_list

def read_weight_file(path):
    vals = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                vals.append(int(line))
    return vals

# ---------------------------------------------------------------
# 1) Generator matrix G from a polynomial P(x) over GF(2)
# ---------------------------------------------------------------
def generator_matrix_from_polynomial(P, n, *, cyclic=False):
    coeffs = list(P)  # low -> high
    degP = P.degree()
    if not cyclic:
        # Non-cyclic Toeplitz-convolution style: row i = coeffs at columns [i .. i+degP]
        k = n - degP
        if k <= 0:
            raise ValueError("For non-cyclic construction, require n > deg(P).")
        rows = []
        for i in range(k):
            row = [0] * n
            for j, c in enumerate(coeffs):
                if i + j < n:
                    row[i + j] ^^= int(c)
            rows.append(row)
        return matrix(F2, rows)
    else:
        base = [0] * n
        for j, c in enumerate(coeffs):
            base[j % n] ^^= int(c)
        rows = []
        cur = base[:]
        for _ in range(n):
            rows.append(cur[:])
            cur = cur[-1:] + cur[:-1]
        G_full = matrix(F2, rows)
        G = G_full.echelon_form()
        G = G.delete_rows([i for i in range(G.nrows()) if G.row(i).is_zero()])
        return G

# ------------------------------------------------------------------
# 2) Systematic form [I_k | A] with column permutation bookkeeping
# ------------------------------------------------------------------
def to_systematic_with_colperm(G):
    """
    Put G into systematic form by (greedy) column swaps + row reduction.
    Returns G_sys, colperm, colperm_inv with colperm[j] = ORIGINAL index of
    the column now at position j.
    """
    n = G.ncols()
    G_work = matrix(G)
    colperm = list(range(n))
    r = 0
    for c in range(n):
        if r >= G_work.nrows():
            break
        if G_work[r, c] == 0:
            found = False
            for c2 in range(c + 1, n):
                if G_work[r, c2] == 1:
                    G_work.swap_columns(c, c2)
                    colperm[c], colperm[c2] = colperm[c2], colperm[c]
                    found = True
                    break
            if not found:
                for r2 in range(r + 1, G_work.nrows()):
                    if G_work[r2, c] == 1:
                        G_work.swap_rows(r, r2)
                        found = True
                        break
                if not found:
                    continue
        if G_work[r, c] == 1:
            for r2 in range(G_work.nrows()):
                if r2 != r and G_work[r2, c] == 1:
                    G_work.add_multiple_of_row(r2, r, 1)
            r += 1
    G_sys = G_work.echelon_form()
    colperm_inv = [0] * n
    for i, p in enumerate(colperm):
        colperm_inv[p] = i
    return G_sys, colperm, colperm_inv

from itertools import combinations
from random import sample, shuffle, getrandbits
from collections import defaultdict
from math import comb, log2, ceil



def apply_colperm_to_vector_fixed(c, colperm):
    n = len(c)
    out = [0] * n
    for j in range(n):
        out[colperm[j]] = int(c[j])
    return vec(out)



def random_systematic_form(G):
    """Returns G_sys, colperm with colperm[j] = original column at position j."""
    n = G.ncols()
    perm0 = list(range(n))
    shuffle(perm0)
    G_perm = G.matrix_from_columns(perm0)
    G_sys, colperm1, _ = to_systematic_with_colperm(G_perm)
    colperm = [perm0[j] for j in colperm1]
    return G_sys, colperm


def _popcount(x):
    return bin(x).count("1")


def _rows_as_ints(G_sys):
    """Row r of G_sys as an int with bit j = G_sys[r, j]."""
    k, n = G_sys.nrows(), G_sys.ncols()
    out = []
    for r in range(k):
        v = 0
        for j in range(n):
            if G_sys[r, j] == 1:
                v |= (1 << j)
        out.append(v)
    return out


def _project(v, cols):
    """Project bit-vector v onto the (ordered) list of columns `cols`, as an int."""
    key = 0
    for i, j in enumerate(cols):
        if (v >> j) & 1:
            key |= (1 << i)
    return key


def _stern_merge(rowsA, rowsB, pa, pb, rowB, rowFull, mask2, target, list_limit):
    """
    Level-2 (base) merge, exactly the Stern collision step:
      all (S1, S2), S1 subset of rowsA with |S1| = pa, S2 subset of rowsB with |S2| = pb,
      such that  (sum_{S1} + sum_{S2}) restricted to B2  ==  target.
    Returns a list of (u_mask, keyB, full_sum) for u = S1 u S2.
    """
    combsA = list(combinations(rowsA, pa)) if pa > 0 else [()]
    combsB = list(combinations(rowsB, pb)) if pb > 0 else [()]
    if len(combsA) > list_limit:
        combsA = sample(combsA, list_limit)
    if len(combsB) > list_limit:
        combsB = sample(combsB, list_limit)

    table = defaultdict(list)
    for S1 in combsA:
        kB = 0
        for r in S1:
            kB ^^= rowB[r]
        table[kB & mask2].append((S1, kB))

    out = []
    for S2 in combsB:
        kB2 = 0
        for r in S2:
            kB2 ^^= rowB[r]
        want = (kB2 ^^ target) & mask2
        if want in table:
            for S1, kB1 in table[want]:
                u = 0
                full = 0
                for r in S1 + S2:
                    u |= (1 << r)
                    full ^^= rowFull[r]
                out.append((u, kB1 ^^ kB2, full))
                if len(out) >= list_limit:
                    return out
    return out


# --------------------------------------------------------------
# 3) BJMM / MMT ISD for codeword search (target weight <= w)
# --------------------------------------------------------------
def bjmm_codeword_search(G, w, *,
                         B_size=20,        # |B| = l, window on which uA must vanish
                         p=4,              # total weight of u on the information part (even)
                         eps=0,            # BJMM overlap parameter (eps = 0  ->  MMT)
                         l2=None,          # size of sub-window B2; default ~ log2(#representations)
                         list_limit=5000,  # cap on every list (memory bound)
                         iters=2000,       # outer iterations (fresh permutation each time)
                         verbose=True):
    """
    Find a low-weight codeword c = u * G of weight <= w using the BJMM
    (depth-2 representation) ISD algorithm. Heuristic; success is probabilistic.

    Returns
    -------
    (success, c, u, wt, stats) — same as stern_codeword_search.
      c  : codeword (vec over GF(2)) in original column order (or None)
      u  : information vector with u * G_sys = c (permuted domain)      (or None)
    """
    k, n = G.nrows(), G.ncols()
    if k == 0:
        raise ValueError("G has zero rows.")
    if w <= 0 or w > n:
        raise ValueError("Target weight w must be in [1..n].")
    if B_size <= 0 or B_size > n - k:
        raise ValueError("B_size must be in [1..n-k].")
    if p <= 0 or p % 2 != 0 or p > k:
        raise ValueError("p must be even and <= k.")
    p1 = p // 2 + eps            # weight of u1, u2
    if 2 * p1 > k:
        raise ValueError("p/2 + eps too large for k.")
    p2a = p1 // 2                # base weights (disjoint halves of rows)
    p2b = p1 - p2a

    # number of representations u = u1 + u2  and default l2 ~ log2(R)
    R = comb(p, p // 2) * comb(k - p, eps)
    if l2 is None:
        l2 = max(0, min(B_size, int(ceil(log2(R))) if R > 1 else 0))
    if l2 > B_size:
        raise ValueError("l2 must be <= B_size.")
    mask2 = (1 << l2) - 1

    stats = dict(iters=0, collisions=0, candidates=0,
                 p=p, eps=eps, l=B_size, l2=l2, representations=R)

    for it in range(1, iters + 1):
        stats['iters'] = it

        # Random column permutation + systematic form
        G_sys, colperm = random_systematic_form(G)
        k_sys, n_sys = G_sys.nrows(), G_sys.ncols()

        # Window B inside the non-pivot ("A") part; B2 = first l2 columns of B
        pivots = set()
        for r in range(k_sys):
            for c in range(n_sys):
                if G_sys[r, c] == 1:
                    pivots.add(c)
                    break
        nonpiv = [c for c in range(n_sys) if c not in pivots]
        B = sample(nonpiv, B_size) if len(nonpiv) >= B_size else sample(range(n_sys), B_size)
        shuffle(B)               # random order -> B2 = B[:l2] is a random sub-window

        rowFull = _rows_as_ints(G_sys)
        rowB = [_project(v, B) for v in rowFull]

        # Random target r on B2 (the representation-technique constraint)
        r_target = getrandbits(l2) if l2 > 0 else 0

        # Level 1: two lists of u's of weight p1 on ALL rows, built Stern-style
        # from two independent random splits of the rows, both with  uA|B2 = r.
        rows = list(range(k_sys))
        shuffle(rows)
        L1 = _stern_merge(rows[:k_sys // 2], rows[k_sys // 2:], p2a, p2b,
                          rowB, rowFull, mask2, r_target, list_limit)
        shuffle(rows)
        L2 = _stern_merge(rows[:k_sys // 2], rows[k_sys // 2:], p2a, p2b,
                          rowB, rowFull, mask2, r_target, list_limit)

        # Level 0 (top): merge L1 x L2 on the FULL window B  (u1A|B == u2A|B,
        # i.e. (u1+u2)A vanishes on B), then filter wt(u) = p and wt(c) <= w.
        table = defaultdict(list)
        for (u1, kB1, f1) in L1:
            table[kB1].append((u1, f1))
            if len(table[kB1]) > 3:            # tiny cap per bucket (as in Stern)
                table[kB1] = table[kB1][-3:]

        for (u2, kB2, f2) in L2:
            if kB2 in table:
                for (u1, f1) in table[kB2]:
                    u = u1 ^^ u2
                    if u == 0 or _popcount(u) != p:   # overlap must be exactly eps
                        continue
                    stats['collisions'] += 1
                    c_int = f1 ^^ f2
                    wt = _popcount(c_int)
                    stats['candidates'] += 1
                    # if wt == w:  #for fixed weight
                    if 1<= wt <= w:
                        c_perm = [ (c_int >> j) & 1 for j in range(n_sys) ]
                        c = apply_colperm_to_vector_fixed(c_perm, colperm)
                        u_vec = vec([ (u >> i) & 1 for i in range(k_sys) ])
                        if verbose:
                            print(f"[iter {it}] success: weight={wt} (target<={w}), "
                                  f"p={p}, eps={eps}, |B|={B_size}, l2={l2}, reps={R}")
                        return True, c, u_vec, wt, stats

        if verbose and it % max(1, iters // 10) == 0:
            print(f"[iter {it}] |L1|={len(L1)}, |L2|={len(L2)}, "
                  f"collisions so far: {stats['collisions']}, candidates: {stats['candidates']}")

    if verbose:
        print("No codeword found within iteration budget.")
    return False, None, None, 0, stats


# --------------------------------------------------------------
# 4) "glue" with the P, PQ from the LWPM step
# --------------------------------------------------------------
def run_bjmm_after_lwpm(P, n=None, *,
                        cyclic=False,
                        w_target=None,
                        B_size=20, p=4, eps=0, l2=None,
                        iters=2000, list_limit=5000, verbose=True):
    """
    Given P from the LWPM solver, build G from P and try to find a codeword
    of weight <= w_target with the BJMM/MMT ISD.
    Also returns Q = c(x) / P(x) (exact division) so that Q is well defined
    even though the rows of G_sys are not the rows of G.
    """
    R = P.parent()
    if n is None:
        n = P.degree() + 1
    G = generator_matrix_from_polynomial(P, n, cyclic=cyclic)

    if verbose:
        print(f"Constructed G of shape {G.nrows()} x {G.ncols()} (cyclic={cyclic}).")
        print(f"Target weight == {w_target}.")

    ok, c, u, wt, stats = bjmm_codeword_search(
        G, w_target, B_size=B_size, p=p, eps=eps, l2=l2,
        list_limit=list_limit, iters=iters, verbose=verbose
    )
    if ok and not cyclic:
        c_poly = R(list(c))
        Q, rem = c_poly.quo_rem(P)
        assert rem == 0
        u = vec(list(Q) + [0] * (G.nrows() - Q.degree() - 1))   # Q as info vector of G
    return ok, c, u, wt, stats, G


# --------------------------------------------------------------
# 5) measurement  + experiment  (same  as for Stern)
# --------------------------------------------------------------
import psutil
from memory_profiler import memory_usage
from random import seed

def run_and_measure_once(f, *args, **kw):
    p = psutil.Process(os.getpid())
    cpu0 = p.cpu_times(); t0 = time.perf_counter()
    mem, out = memory_usage((f, args, kw), interval=0.05, max_iterations=1, retval=True)
    t1 = time.perf_counter(); cpu1 = p.cpu_times()
    return {"result": out, "wall_time_s": t1 - t0,
            "cpu_time_s": (cpu1.user - cpu0.user) + (cpu1.system - cpu0.system),
            "peak_mem_MiB": max(mem)}

def worker_function(queue, func, *args, **kwargs):
    try:
        queue.put(("success", run_and_measure_once(func, *args, **kwargs)))
    except Exception:
        queue.put(("error", traceback.format_exc()))

def run_with_timeout(timeout_sec, func, *args, **kwargs):
    queue = multiprocessing.Queue()
    p = multiprocessing.Process(target=worker_function, args=(queue, func, *args), kwargs=kwargs)
    p.start(); p.join(timeout=timeout_sec)
    if p.is_alive():
        p.terminate(); p.join()
        return {"status": "timeout", "message": f"No solution found within {timeout_sec} seconds."}
    if not queue.empty():
        status, data = queue.get()
        return {"status": "success", "data": data} if status == "success" else {"status": "error", "message": data}
    return {"status": "error", "message": "Unknown error."}


if __name__ == '__main__':
    in_dir    = r""
    pq_in_dir = r""
    p_path  = os.path.join(in_dir, "coeffs_p.txt")
    pq_path = os.path.join(pq_in_dir, "pq_norm.txt")

    out_dir = r""
    os.makedirs(out_dir, exist_ok=True)

    t = 400  # Degree of P
    d = 100  # Degree of Q
    w = 10  # Max Hamming weight
    n=500

    P_all  = read_P_file(p_path)
    PQ_all = read_weight_file(pq_path)

    for i, (P, PQ) in enumerate(zip(P_all, PQ_all)):
        with open(os.path.join(out_dir, "pq.txt"), "a+") as f_PQ, \
             open(os.path.join(out_dir, "pq_norm.txt"), "a+") as f_norm, \
             open(os.path.join(out_dir, "q.txt"), "a+") as f_Q, \
             open(os.path.join(out_dir, "wall_time.txt"), "a+") as f_wall, \
             open(os.path.join(out_dir, "cpu_time.txt"), "a+") as f_cpu, \
             open(os.path.join(out_dir, "peak_mem.txt"), "a+") as f_mem:

            print(f"Instance {i + 1}/{len(P_all)}")
            seed(i)
            P = list_to_poly(P)
            print("P=", P); print("Pq=", PQ)

            try:
                result = run_with_timeout(
                    120, run_bjmm_after_lwpm,
                    P=P, n=n, cyclic=False, w_target=PQ,
                    B_size=10, p=4, eps=0, l2=None,
                    iters=200, list_limit=5000, verbose=True)

                if result["status"] == "timeout":
                    print("Timeout.")
                    for f in (f_wall, f_cpu, f_mem, f_norm): f.write("TIMEOUT\n")
                    f_PQ.write(str([0]) + "\n"); f_Q.write(str([0]) + "\n")
                    continue
                elif result["status"] == "error":
                    print(result["message"]); continue

                measure = result["data"]
                ok, c, u, wt, stats, G = measure["result"]
                print(f"Wall time   : {measure['wall_time_s']:.3f} s")
                print(f"CPU time    : {measure['cpu_time_s']:.3f} s")
                print(f"Peak memory : {measure['peak_mem_MiB']:.1f} MiB")
                f_wall.write(f"{measure['wall_time_s']}\n")
                f_cpu.write(f"{measure['cpu_time_s']}\n")
                f_mem.write(f"{measure['peak_mem_MiB']}\n")

                if ok:
                    print("Found codeword with weight", wt)
                    print("c =", R(list(c)))
                    print("Q =", R(list(u)))
                    f_PQ.write(str(c) + "\n"); f_norm.write(str(wt) + "\n"); f_Q.write(str(u) + "\n")
                else:
                    print("No improvement within the given budget.")
                    f_PQ.write(str([0]) + "\n"); f_norm.write("0\n"); f_Q.write(str([0]) + "\n")
            except Exception as e:
                print(e); print("Solver error, skipping instance", i); continue
